# The Agent Protocol Stack

## Northstar: delegate incident analysis across boundaries

A coordinator discovers a remote release-analysis agent, delegates a tenant-scoped task, obtains deployment data through a narrow tool contract, and presents a human approval card. We use it to distinguish MCP, A2A, AG-UI, A2UI, commerce, and payment boundaries.

**Outcomes:** explain why no one protocol replaces the others; inspect A2A discovery/delegation; model narrow MCP tool access; distinguish interaction from generated UI; and maintain identity, policy, audit, and authorization outside protocol metadata. All examples are deterministic and credential-free.

![Agent protocol stack](../../../assets/agent-protocol-stack.svg)

The stack is a teaching model, not a normative standard. Identity, delegated authority, policy, observability, and audit span every layer; they are not provided automatically by an agent message, tool schema, or UI card.

## Step 1 — map the boundary before choosing a protocol

**MCP** is agent/client ↔ tools/resources/prompts/context. **A2A** is agent ↔ agent task collaboration. **AG-UI** is agent ↔ application interaction events. **A2UI** is structured agent-generated UI description ↔ native renderer. **UCP/AP2** address commerce/payment interaction and intent. A REST endpoint can still be the right choice; protocols earn adoption when interoperable capability, discovery, lifecycle, or renderer contracts remove repeated bespoke integration without weakening controls.

In [1]:
from pathlib import Path
import sys
TOPIC = Path.cwd()
if not (TOPIC / 'lab.py').exists():
    TOPIC = Path.cwd() / 'curriculum' / 'enterprise-agent' / '17-agent-protocol-stack'
sys.path.insert(0, str(TOPIC))
from lab import AgentCard, Task, delegate, discover, mcp_tool_call, ui_event

cards = [
    AgentCard('release-agent', ('deployment-analysis',), 'northstar-eu', 'oauth'),
    AgentCard('other-tenant-agent', ('deployment-analysis',), 'other-tenant', 'oauth'),
]
eligible = discover(cards, 'deployment-analysis', 'northstar-eu')
print(eligible)
assert [card.name for card in eligible] == ['release-agent']

[AgentCard(name='release-agent', skills=('deployment-analysis',), tenant='northstar-eu', auth='oauth', risk='read')]


## Step 2 — A2A discovery and delegation

A2A treats a remote agent as a collaborator with a task lifecycle, not merely a function. An Agent Card describes capabilities and authentication requirements; discovery finds candidates; a task can receive messages/status and be cancelled. That is useful for remote, cross-framework, long-running, or opaque specialists.

But an Agent Card is not trust or authorization. An enterprise must verify issuer/registry, identity/authentication, tenant/residency, capability version, risk, cost/SLO, policy and revocation before selecting it. The delegated task must minimize data and include an ID, objective, allowed scope, deadline, budget, expected artifact, cancellation, and correlation ID.

In [2]:
task = Task('task-17', 'deployment-analysis', 'northstar-eu')
assert delegate(eligible[0], task) == 'working'
print(task)

try:
    delegate(AgentCard('wrong', ('deployment-analysis',), 'other-tenant', 'oauth'), task)
except ValueError as error:
    print('blocked:', error)

assert task.trace == ['a2a:delegated:release-agent']

Task(task_id='task-17', skill='deployment-analysis', tenant='northstar-eu', status='working', trace=['a2a:delegated:release-agent'])
blocked: capability/scope mismatch


## Step 3 — MCP: a narrow tool/context boundary

MCP servers expose tools, resources, and prompts through schemas. The agent/client may ask to invoke a tool; application policy decides whether the caller has the required scope and whether the result is trusted as data. Tool results may contain wrong or adversarial content, so they do not supersede system policy. Discovery should be authorization-aware; do not expose a global high-privilege tool catalogue to every request.

The simulator allows only `read_deployment` with a read scope. A production tool boundary needs typed arguments/results, tenant checks, idempotency for mutations, rate/budget limits, traces, and human approval for consequential operations.

In [3]:
print(mcp_tool_call('read_deployment', {'deployments.read'}))
try:
    mcp_tool_call('read_deployment', set())
except PermissionError as error:
    print('denied:', error)

assert mcp_tool_call('read_deployment', {'deployments.read'})['trusted_as'] == 'data-not-instructions'

{'tool': 'read_deployment', 'result': 'deploy-842', 'trusted_as': 'data-not-instructions'}
denied: MCP tool denied


## Step 4 — AG-UI versus A2UI

AG-UI carries agent/application lifecycle and interaction events, supporting streaming, state updates, and interruptible runs. A2UI describes structured components a native renderer may display. The distinction matters: interaction events do not grant backend authority, and a schema-rendered component should not execute arbitrary code.

A human approval card may offer `approve`, `reject`, or `modify`; after the click, the server authenticates the person and rechecks request scope, action fingerprint, freshness, policy, and idempotency. The UI cannot be a bypass around action authorization.

In [4]:
event = ui_event('approval-card', 'approve')
print(event)
try:
    ui_event('approval-card', 'execute-arbitrary-javascript')
except ValueError as error:
    print('schema blocked:', error)
assert event['requires_reauthorization'] is True

{'ag_ui_event': 'approve', 'component': 'approval-card', 'requires_reauthorization': True}
schema blocked: UI action not in schema


## Step 5 — commerce and payment boundaries

UCP/AP2-style contracts are relevant when agents assist with product discovery, carts, checkout, payment intent, or delegated transaction authorization. They do **not** remove merchant/provider controls. Use a transaction service and explicit user intent; isolate credentials from context; apply spend limits, approvals, risk/fraud controls, receipts, reconciliation, and revocation. The ecosystem is evolving, so integrate behind adapters and validate the current specification and provider requirements before production use.

## Protocol selection and production exercises

- Use **MCP** for a constrained context/tool service.
- Use **A2A** when a remote agent needs discoverable capability, task lifecycle, messaging, delegation, and status/cancel semantics.
- Use **AG-UI/A2UI** to standardize user interaction and dynamic approved components—not backend authority.
- Use commerce/payment adapters only with separate identity, consent, policy, and financial control planes.

**Exercises:** add an expired A2A task; reject a revoked Agent Card; scope an MCP tool catalogue by tenant; model an A2UI component allowlist; add a transaction proposal that cannot execute without a distinct human/provider authorization.

References: [MCP](https://modelcontextprotocol.io/specification/), [A2A](https://a2a-protocol.org/latest/), [AG-UI](https://docs.ag-ui.com/), [A2UI](https://a2ui.org/specification/v0.9-a2ui/), [interoperability survey](https://arxiv.org/abs/2505.02279).